In [2]:
import xwrf
import glob
import pickle

import xarray as xr
import matplotlib.pyplot as plt
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import numpy as np
from wrf import latlon_coords, ll_to_xy, xy_to_ll
from netCDF4 import Dataset
from wrf import getvar #for RH

Extract temperature and relative humidity from WRF files at the station locations - to be used for Figure 4

In [ ]:
#identify URBAN-ONLY stations
# Define file path and domain boundaries
file_luindex = "/g/data/fy29/mf9078/run_WRF/etamodified/output/4out/animation/wrfout_d02_2017-01-12_21:00:00"

# mask
lat_min, lat_max = -34.14, -33.55
lon_min, lon_max = 150.57, 151.37

#14 DCCEEW stations in Sydney
slat = [-33.93175,-33.86433,-33.78113,-33.93132,-33.91766,-33.89156,-33.91619,-33.61641,-34.30621,-33.79512,-34.05170,-33.79424,-34.05610,-34.04168]
slon = [151.24278,151.16395,151.15090,150.90727,150.76192,151.04610,151.13577,150.74731,150.58061,150.76677,150.49819,150.91417,150.81220,150.69013]
snm = ['RK','RE','LD','LL','BY','CA','ED','RD','BO','SS','OE','PT','CN','CD']
stations = {'RK':'Randwick','RE':'Rozelle','LD':'Lindfield','LL':'Liverpool','BY':'Bringelly','CA':'Chullora','ED':'Earlwood','RD':'Richmond','BO':'Bargo','SS':'St Marys','OE':'Oakdale','PT':'Prospect','CN':'Campbelltown','CD':'Camden'}

snm_indomain=[]
slat_indomain=[]
slon_indomain=[]
# Add station labels
for i, (lon, lat, name) in enumerate(zip(slon, slat, snm)):
    if (lat >= lat_min) & (lat <= lat_max) & (lon >= lon_min) & (lon <= lon_max):
        snm_indomain.append(name)
        slat_indomain.append(lat)
        slon_indomain.append(lon)
    #else:
        #print(f"Station {name} falls outside the domain")

#xy of WRF grids corresponding to the stations locations within the latlon limit
x_y = ll_to_xy(Dataset(file_luindex), slat_indomain, slon_indomain, stagger=None, as_int=True)
sn = x_y[1].values
we = x_y[0].values

#lat/lon of WRF grids corresponding to the stations locations
l_l = xy_to_ll(Dataset(file_luindex), we, sn, stagger=None)
slat_wrfgrids=l_l[0].values
slon_wrfgrids=l_l[1].values

#xy and ll of WRF grids corresponding to the stations locations - ONLY URBAN
ds = xr.open_dataset(file_luindex)

snm_urban=[]
sn_urban=[]
we_urban=[]
slat_urban=[]
slon_urban=[]

for i in range(len(snm_indomain)):
    if ds['LU_INDEX'].isel(south_north=sn[i], west_east=we[i]).item() > 50:
        snm_urban.append(snm_indomain[i])
        sn_urban.append(sn[i])
        we_urban.append(we[i])      
             
#lat/lon of WRF grids corresponding to the stations locations - URBAN ONLY
l_l_urban = xy_to_ll(Dataset(file_luindex), we_urban, sn_urban, stagger=None)
slat_wrfurban=l_l_urban[0].values
slon_wrfurban=l_l_urban[1].values


In [4]:
snm_urban

['RK', 'RE', 'LL', 'CA', 'ED', 'SS', 'PT', 'CN']

In [5]:
def calculate_relative_humidity(PSFC, Q2, T2):
    #formula from https://forum.mmm.ucar.edu/threads/relative-humidity.9134/
    #constants
    svp1=611.2
    svp2=17.67
    svp3=29.65
    svpt0=273.15
    eps = 0.622
    
    #calculate RH
    rh = 1.E2 * (PSFC*Q2/(Q2*(1.-eps) + eps))/(svp1*np.exp(svp2*(T2-svpt0)/(T2-svp3)))
    return rh
#note: rh2 = getvar(ncfile, "rh2").values  doesn't work with wrfxtrm files

In [ ]:
#load surface pressure data of WRF - see the bottom for their extraction
with open("/g/data/gb02/mf9078/WRF_pavg-10-20Jan_DCCEEW_mean.pkl", "rb") as f:
    wrf_stations_pavg = pickle.load(f)

In [9]:
#extract T2 at station locations for all cases

scenario = ['Default(Bulk+NoUCM)', 'LCZ', 'WSF-MB', 'Geoscape']
wrf_files = [
    "/g/data/fy29/mf9078/run_WRF/etamodified/output/0mean/wrfxtrm_d02_2017-01*", #0out/animation/wrfout_d02_2017-01*
    "/g/data/fy29/mf9078/run_WRF/etamodified/output/1mean/wrfxtrm_d02_2017-01*",
    "/g/data/fy29/mf9078/run_WRF/etamodified/output/3mean/wrfxtrm_d02_2017-01*", 
    "/g/data/fy29/mf9078/run_WRF/etamodified/output/4mean/wrfxtrm_d02_2017-01*"   
]

wrf_stations = {}
wrf_s_q = {}
wrf_s_rh = {}
for sc in scenario:
    wrf_stations[sc] = {}
    wrf_s_q[sc] = {}
    wrf_s_rh[sc] = {}
    
for j in range(len(wrf_files)):
    files = sorted(glob.glob(wrf_files[j]))
    if j == 0: #any set of files
        ds = xr.open_mfdataset(files, parallel=True, concat_dim="Time", combine="nested").xwrf.postprocess()
        sydney_time = ds["Time"] + np.timedelta64(10, "h")  # Convert UTC to Sydney time AEST
        wrf_stations['time'] = sydney_time
        wrf_s_rh['time'] = sydney_time

    # Initialize keys with empty lists
    temp = {}
    q = {}
    for name in snm_urban:
        temp[name] = []
        q[name]=[]

    for file in files:
        ds = xr.open_dataset(file)

        for i in range(len(snm_urban)):
            temp[snm_urban[i]].append(ds['T2MEAN'].isel(south_north=sn_urban[i], west_east=we_urban[i]).item() - 273.15)
            q[snm_urban[i]].append(ds['Q2MEAN'].isel(south_north=sn_urban[i], west_east=we_urban[i]).item())

    wrf_stations[scenario[j]]=temp
    wrf_s_q[scenario[j]]=q

for sc in scenario:
    for station in snm_urban:
        rh = []
        for i in range(len(wrf_stations_pavg['LCZ']['RK'])): #any list
            t_kelvin = wrf_stations[sc][station][i] + 273.15 
            rh.append(calculate_relative_humidity(wrf_stations_pavg[sc][station][i], wrf_s_q[sc][station][i], t_kelvin))
        
        wrf_s_rh[sc][station] = rh
    
with open("/g/data/gb02/mf9078/WRF_Results-10-20Jan_xy_wtime_DCCEEW_mean.pkl", "wb") as f:
    pickle.dump(wrf_stations, f)

with open("/g/data/gb02/mf9078/WRF_Results-10-20Jan_DCCEEW_RH_mean.pkl", "wb") as f:
    pickle.dump(wrf_s_rh, f)

In [ ]:
#calculate average over the 8 stations for each scenario
scenario = ['Default(Bulk+NoUCM)', 'LCZ', 'WSF-MB', 'Geoscape']  
average_t = {}
average_rh = {}
for sc in scenario:
    average_t[sc] = [] 
    average_rh[sc] = [] 

for sc in scenario:
    for i in range(len(wrf_stations['Default(Bulk+NoUCM)']['RK'])): #any list
        t_values = [wrf_stations[sc][station][i] for station in snm_urban]
        average_t[sc].append(sum(t_values) / len(t_values))

        rh_values = [wrf_s_rh[sc][station][i] for station in snm_urban]
        average_rh[sc].append(sum(rh_values) / len(rh_values))

average_t['time'] = wrf_stations['time']
average_rh['time'] = wrf_s_rh['time']

with open("/g/data/gb02/mf9078/WRF_Results-10-20Jan_xy_avg_DCCEEW_mean.pkl", "wb") as f:
    pickle.dump(average_t, f)

with open("/g/data/gb02/mf9078/WRF_Results-10-20Jan_avgDCCEEW_RH_mean.pkl", "wb") as f:
    pickle.dump(average_rh, f)

In [12]:
len(average_rh['Geoscape'])

241

Extract PSFC at station locations for all cases 

In [ ]:
#PSFC is not among the variables that WRF can save as hourly mean, so I calculated PSFC_avg = (PSFC_t-1 + PSFC_t)/2 from instantaneous values to get a value to the average of the previous hour. 
#added 1 wrfout file for Jan 8th, 23:00 for getting the avg value for Jan 9th 00:00 
# => executed only once and saved the results for future use 

scenario = ['Default(Bulk+NoUCM)', 'LCZ', 'WSF-MB', 'Geoscape']
wrf_files = [
    "/g/data/fy29/mf9078/run_WRF/etamodified/output/duplicate/0out/wrfout_d02_2017-01*",
    "/g/data/fy29/mf9078/run_WRF/etamodified/output/duplicate/1out/wrfout_d02_2017-01*",
    "/g/data/fy29/mf9078/run_WRF/etamodified/output/duplicate/3out/wrfout_d02_2017-01*",
    "/g/data/fy29/mf9078/run_WRF/etamodified/output/duplicate/4out/wrfout_d02_2017-01*"  
]

wrf_stations_p = {}
for sc in scenario:
    wrf_stations_p[sc] = {}
    
for j in range(len(wrf_files)):
    files = sorted(glob.glob(wrf_files[j]))
    if j == 0: #any set of files
        ds = xr.open_mfdataset(files, parallel=True, concat_dim="Time", combine="nested").xwrf.postprocess()
        sydney_time = ds["Time"] + np.timedelta64(10, "h")  # Convert UTC to Sydney time AEST
        wrf_stations_p['time'] = sydney_time
        
    # Initialize keys with empty lists
    p = {}
    
    for name in snm_urban:
        p[name] = []
        
    for file in files:
        ds = xr.open_dataset(file)

        for i in range(len(snm_urban)):
            p[snm_urban[i]].append(ds['PSFC'].isel(south_north=sn_urban[i], west_east=we_urban[i]).item() - 273.15)
            
    wrf_stations_p[scenario[j]]=p

wrf_stations_pavg = {}
for sc in scenario:
    wrf_stations_pavg[sc]={}
    for station in snm_urban:
        wrf_stations_pavg[sc][station]=[]
        for i in range(len(wrf_stations_p['LCZ']['RK'])-1): #any of the lists
            avg= (wrf_stations_p[sc][station][i]+wrf_stations_p[sc][station][i+1])/2
            wrf_stations_pavg[sc][station].append(avg)

with open("/g/data/gb02/mf9078/WRF_pavg-10-20Jan_DCCEEW_mean.pkl", "wb") as f:
    pickle.dump(wrf_stations_pavg, f)
